# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We’ll print the top-level record sets defined in the dataset using their `@id`. Then for each record set, list its fields (columns) and their `@id` for downstream reference.

In [ ]:
# List all available record sets with their @id
print("Record Sets (with @id):")
for rs in metadata.record_sets:
    print(f"  - {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'field' in rs:
        print(f"    Fields (with @id):")
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            print(f"      * {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we select the first available record set from above.

record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        # Store only if records were found
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
    else:
        print(f"No records found for record set: {record_set_id}")

# Choose the first non-empty record set for EDA
main_record_set_id = next(iter(dataframes)) if dataframes else None

if main_record_set_id:
    print(f"\nExample columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded. Please check the dataset schema for available data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Select a numeric field (e.g., 'cr:coverage_percent' or similar, identified by its `@id`), filter records by a threshold, and normalize the values.

In [ ]:
# Replace these with available numeric field(s) from your chosen record set.
# For demonstration, infer a numeric column (e.g. 'cr:coverage_percent', 'cr:MW', etc.)
df = dataframes[main_record_set_id]

# Try to guess a numeric field by looking at the columns ending with common numeric patterns
import re
numeric_field_candidates = [col for col in df.columns if re.search(r'(percent|count|MW|weight|abundance|number|int|float)', col, re.IGNORECASE)]
print(f"Numeric candidate fields: {numeric_field_candidates}")

numeric_field = None
for c in numeric_field_candidates:
    # Try to check column type
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field = c
        break

# If no candidates, use the first numeric column found
if numeric_field is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

print(f"Selected numeric field: {numeric_field}")

if numeric_field:
    # Drop NaN or non-numeric
    thresh = df[numeric_field].dropna().median() if df[numeric_field].dropna().size else 10
    filtered_df = df[df[numeric_field] > thresh]
    print(f"Filtered records with {numeric_field} > {thresh}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Search for a grouping field
    group_field_candidates = [col for col in df.columns if re.search(r'(group|type|class|category|sample)', col, re.IGNORECASE)]
    print(f"Possible group-by fields: {group_field_candidates}")

    group_field = group_field_candidates[0] if group_field_candidates else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field was found in the data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field, and if grouped data is available, show a bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.
- Loaded metadata and inspected structure using `@id` for all record sets and fields.
- Extracted a sample record set into a pandas DataFrame, identified available numeric and grouping fields using dynamic detection.
- Demonstrated basic EDA: filtering, normalization, grouping, and visualization.

You may now further analyze the dataset, perform advanced modeling, or apply additional transformations relevant to your research question.